In [ ]:
import dataclasses
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import pandas
import named_arrays as na
import optika
import esis

In [ ]:
instrument = esis.flights.f2.optics.design_single(num_distribution=0)

In [ ]:
instrument.field.num = 5

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        figsize=(8, 4),
        constrained_layout=True,
    )
    instrument.system.plot(
        ax=ax,
        components=("z", "x"),
        color="black",
        kwargs_rays=dict(
            color=na.ScalarArray(
                ndarray=np.array(["tab:blue", "tab:orange"]),
                axes="wavelength",
            ),
            label=instrument.wavelength.to_string_array(),
            zorder=-instrument.wavelength.value,
        ),
    )
    ax.set_xlabel(f"$z$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"$x$ ({ax.get_ylabel()})")

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())

In [ ]:
instrument.system.spot_diagram();

In [ ]:
position = na.Cartesian2dVectorArray(
    x=na.linspace(0, 80, axis="x", num=11) * u.mm,
    y=0 * u.mm,
)

In [ ]:
z = -instrument.primary_mirror.sag(position)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(position.x, z, label="primary mirror")
    ax.set_xlabel(f"x ({ax.get_xlabel()})")
    ax.set_ylabel(f"sag ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
print(pandas.DataFrame(np.stack([position.x.ndarray, z.ndarray]).T, columns=["x", "sag"]))

In [ ]:
field = na.Cartesian2dVectorLinearSpace(
    start=-0.99,
    stop=+0.99,
    axis=na.Cartesian2dVectorArray("fx", "fy"),
    num=11,
)

In [ ]:
pupil=na.Cartesian2dVectorLinearSpace(
    start=-0.99,
    stop=+0.99,
    axis=na.Cartesian2dVectorArray("px", "py"),
    num=41,
)

In [ ]:
instrument_full = esis.flights.f2.optics.design(
    grid=optika.vectors.ObjectVectorArray(
        wavelength=instrument.wavelength,
        field=field,
        pupil=pupil,
    ), 
    num_distribution=0,
)

In [ ]:
instrument_full.roll = 22.5 * u.deg

In [ ]:
transformation = na.transformations.Cartesian3dRotationY(180 * u.deg)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    instrument_full.schematic_primary(
        ax=ax,
        transformation=transformation,
    )
    ax.set_aspect("equal")
    ax.legend()